
---

## **Part 1 — Exploring Different Modalities / Representations of Network Traffic**

Synthetic network traffic generation is useful for many applications such as dataset augmentation, network testing, and resource management. Many existing generation methods treat traffic generation as either a time-series prediction task or an autoregressive modeling task. In these approaches, models are trained directly on structured representations of packets—one common example is **nPrint**, a tabular format that encodes packet header fields from PCAP traces into machine-learning-friendly numerical vectors.

In an **nPrint**, each row represents one packet in a trace. The entire nPrint file (a large CSV) represents all packets of that trace in sequential order. Header fields are encoded using one-hot–like binary indicators (e.g., `1`, `0`, or `01`), making them easy to feed into ML models.

However, general-purpose ML models—even advanced time-series architectures and transformers—often struggle to capture the *complex dependencies* present in real network traffic:

* **Local dependencies**: relationships among columns within a single row (i.e., dependencies among header fields of a single packet).
* **Global dependencies**: relationships across rows (i.e., the evolution of packets within the same flow).
  For example: if the first packet of a flow uses TCP, the second packet in that same flow should also be TCP; sequence numbers, flags, and flow identifiers evolve in structured ways over time.

While sequential ML models struggle with these multi-scale dependencies, **vision models** (e.g., diffusion models) excel at capturing both local and global structure when data is presented spatially—like an image. By converting nPrint traces into 2D PNG images, we can take advantage of the strong representational capabilities of image models and generate synthetic traffic using visual generative approaches.

---

### **Your Task for Part 1**

In this part of the assignment, you will:

1. **Inspect the raw nPrint files** (found in the `real_nprints` directory).
2. **Understand how each CSV row and column corresponds to packet-level metadata.**
3. **Follow the provided conversion pipeline** that transforms these nPrint CSVs into PNG image representations suitable for use with diffusion models and other visual architectures.
4. **Take an already generated set of image-representation of images and convert them back into nprint representation for downstream task utilization**

This will help you understand why converting network traces to images can unlock generative modeling capabilities that traditional ML approaches struggle with.

Q1:
First, download and unzip the data you will need from (https://drive.google.com/file/d/1hY6nNXEYOwl1l-O_nCknO9xezcHr6ZXi/view?usp=sharing)
In your own words, describe how an nPrint CSV encodes a network trace.
Why might a 2D image representation capture structural relationships that a row-by-row CSV cannot?

An nPrint CSV encodes a packet capture by alligning the byte of the trace to a standardized format. Each column represents a sepcific feature and can be filled with a byte, 0, 1, -1, to indicate if the feature appears in the file or not. Because every trace uses the same fixed column layout (IPV4, TCP, UDP, ICMP, Payload features), traces of different lengths can be compared and fed into ML models. 

In an image, bits within a packet are horizontal neighbors and consecutive packets are verticle neighbors. Vision models and diffusion models are designed to capture these kinds of local and spactial patters through 2D convolution. A row-by-row CSV model can learn cross row relationships, but it must infer them from sequential process rather than having them built into the representation. The image layout makes both within packet anda cross-packet structure explicit at once.

Q2. Design a method for converting nPrint representations of traces (in the folder real_nprints) into image representations.
Your image representation should use only the first 1024 packets from each trace to avoid producing images that are too large.
Save the images into a folder called './nd_data/student_converted_images'

Method Description

1. Load the nprint file using pandas. 

2. Select the first 1024 rows. Since a packet is represented by a row and we want to use the first 1024.

3. Map each cell value to a RGB pixel color
   - `1`  → red   `(255, 0, 0, 255)`   — bit is set
   - `0`  → green `(0, 255, 0, 255)`   — bit is unset
   - `-1` → blue  `(0, 0, 255, 255)`   — field not present in this packet

4. Construct a image from the represenatons 
   - **Width** = number of nPrint columns (header features)
   - **Height** = 1024 (fixed, one row per packet)
   - If a trace has fewer than 1024 packets, pad the remaining rows with blue (`-1`) so all images have the same size for the diffusion model.

This preserves local structure (header bits within a packet are horizontal neighbors) and temporal structure (consecutive packets are vertical neighbors), which is what vision models uses.

In [1]:
import os
import glob
import numpy as np
import pandas as pd
from PIL import Image

INPUT_DIR = "./nd_data/real_nprints"
OUTPUT_DIR = "./nd_data/student_converted_images"
MAX_PACKETS = 1024

# Map nPrint values to RGBA pixels
def value_to_rgba(v):
    v = int(v)
    if v == 1:
        return (255, 0, 0, 255)      # red = bit set
    elif v == 0:
        return (0, 255, 0, 255)      # green = bit unset
    elif v == -1:
        return (0, 0, 255, 255)      # blue = field not present
    else:
        return (0, 0, 0, 255)          # fallback

def nprint_to_image(df, max_packets=1024):
    """Convert nPrint DataFrame to a fixed-size RGBA image."""
    # Keep only first 1024 packets (rows)
    df = df.iloc[:max_packets].copy()

    width = df.shape[1]
    height = len(df)

    # Start with all-blue padding (-1) for unused packet rows
    img_array = np.full((max_packets, width, 4), (0, 0, 255, 255), dtype=np.uint8)

    # Fill in actual packet rows
    for row_idx in range(height):
        for col_idx in range(width):
            img_array[row_idx, col_idx] = value_to_rgba(df.iat[row_idx, col_idx])

    return Image.fromarray(img_array, mode="RGBA")

def convert_directory(input_dir, output_dir, max_packets=1024):
    os.makedirs(output_dir, exist_ok=True)

    ip_cols = ["ipv4_src", "ipv4_dst", "ipv6_src", "ipv6_dst", "src_ip"]
    nprint_files = glob.glob(os.path.join(input_dir, "*.nprint"))

    for path in nprint_files:
        filename = os.path.basename(path)
        print(f"Processing {filename}")

        df = pd.read_csv(path)

        # Drop row index column if present
        if df.columns[0].lower().startswith("unnamed") or df.columns[0] == "":
            df = df.iloc[:, 1:]

        # Drop high-cardinality IP address columns
        df = df.drop(
            columns=[c for c in df.columns if any(s in c for s in ip_cols)],
            errors="ignore",
        )

        img = nprint_to_image(df, max_packets=max_packets)

        out_path = os.path.join(
            output_dir, filename.replace(".nprint", ".png")
        )
        img.save(out_path)

    print(f"\nDone! Saved {len(nprint_files)} images to {output_dir}")

# Run conversion
convert_directory(INPUT_DIR, OUTPUT_DIR, MAX_PACKETS)

Processing teams_1024_10.nprint
Processing twitch_1024_7.nprint
Processing netflix_1024_15.nprint
Processing twitter_1024_17.nprint
Processing netflix_1024_19.nprint
Processing meet_1024_18.nprint
Processing youtube_1024_6.nprint
Processing netflix_1024_2.nprint
Processing teams_1024_3.nprint
Processing zoom_1024_8.nprint
Processing meet_1024_1.nprint
Processing instagram_1024_3.nprint
Processing zoom_1024_4.nprint
Processing youtube_1024_11.nprint
Processing meet_1024_14.nprint
Processing facebook_1024_6.nprint
Processing instagram_1024_19.nprint
Processing instagram_1024_15.nprint
Processing twitch_1024_17.nprint
Processing amazon_1024_17.nprint
Processing zoom_1024_13.nprint
Processing facebook_1024_17.nprint
Processing teams_1024.nprint
Processing amazon_1024.nprint
Processing twitch_1024_15.nprint
Processing amazon_1024_2.nprint
Processing instagram_1024_17.nprint
Processing facebook_1024_8.nprint
Processing twitch_1024.nprint
Processing facebook_1024_4.nprint
Processing twitch_10

In [2]:
# The following is a pre-defined script used in NetDiffusion that will convert all of the provided real nprints into image representations. Run this code and observe the output
!python ./scripts/nprint_to_png.py -i ./nd_data/real_nprints/ -o ./nd_data/real_traffic_images

Processing teams_1024_10.nprint
/Users/aeliyagrover/Documents/GitHub/ml-systems/completed assignments/netdiffusion/./scripts/nprint_to_png.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  np_df = np.array(df.applymap(np.array).to_numpy().tolist())
Processing twitch_1024_7.nprint
/Users/aeliyagrover/Documents/GitHub/ml-systems/completed assignments/netdiffusion/./scripts/nprint_to_png.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  np_df = np.array(df.applymap(np.array).to_numpy().tolist())
Processing netflix_1024_15.nprint
/Users/aeliyagrover/Documents/GitHub/ml-systems/completed assignments/netdiffusion/./scripts/nprint_to_png.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  np_df = np.array(df.applymap(np.array).to_numpy().tolist())
Processing twitter_1024_17.nprint
/Users/aeliyagrover/Documents/GitHub/ml-systems/completed assignments/netdiffusion/./scripts/n

Q3: Now that you have seen how NetDiffusion converts nPrints into images, compare their method with the approach you designed in Q2. What are the advantages and disadvantages of each, especially in terms of what might help or hinder a vision model’s ability to learn?

### Similarities
Both my method and NetDiffusion's script use the same core idea:
- Each **row** = one packet, each **column** = one nPrint header bit
- Map cell values to **RGBA colors**: `1→red`, `0→green`, `-1→blue`
- **Fixed image height** of 1024 rows, padding shorter traces with blue (`-1`)
- **Drop IP address columns** to remove high-cardinality identifiers that don't generalize well

This spatial layout helps vision models because header-bit dependencies are horizontal neighbors and packet-sequence dependencies are vertical neighbors — exactly the kind of local structure CNNs and diffusion models capture well.

### My approach (Q2) — Advantages
- **Drops the row index column** before conversion, so every pixel corresponds to a meaningful header bit. My images are 1088×1024 vs NetDiffusion's 1089×1024 — the extra column in theirs encodes packet numbers (0, 1, 2…) as colors, which isn't semantically useful and could add noise for the model.
- **Explicitly truncates to the first 1024 packets**, guaranteeing consistent input size regardless of trace length.
- **Simple, readable mapping** — only three discrete colors, making the image→nPrint reverse conversion straightforward.

### My approach — Disadvantages
- **Slower implementation** (nested Python loops vs their vectorized `apply`), though this doesn't affect model learning, only preprocessing time.
- **Less robust value handling** — I only map `1`, `0`, and `-1`, while NetDiffusion also handles edge-case values (`>1`, `<-1`) using the alpha channel.
- **No error handling** for malformed files.

### NetDiffusion's approach — Advantages
- **Vectorized conversion** is faster and more production-ready.
- **Handles unusual cell values** beyond `{0, 1, -1}` via alpha-channel encoding, which preserves more information if non-standard values appear.
- **Includes error handling** and duplicate-filename protection.
- This is the **exact pipeline used to train their diffusion model**, so using `real_traffic_images/` ensures consistency with the pretrained model.

### NetDiffusion's approach — Disadvantages
- **Keeps the index column**, adding a leftmost strip of pixels that encode row numbers rather than packet header semantics. A vision model might waste capacity learning patterns in this column that don't transfer to generation quality.
- **Does not explicitly truncate** to 1024 rows — it relies on input files already being the right length.

### Impact on vision model learning
Both methods should work well for diffusion because they preserve the key spatial structure. The biggest learning-relevant difference is the **index column**: removing it (my approach) gives a cleaner, more semantically uniform image where every pixel represents a header bit. Keeping it (NetDiffusion) adds a consistent but non-semantic column that the model must learn to ignore or exploit.

The shared strengths — fixed dimensions, discrete color encoding, IP removal, and 2D spatial layout — are what matter most for enabling vision models to learn traffic patterns. The differences are mostly about preprocessing cleanliness and robustness rather than fundamentally different representations.


---

## **Part 2 — Converting Generated Images Back Into Usable Format**

In the first part of the assignment, you explored how network traces in nPrint format can be transformed into image representations suitable for vision-based generative models. In Part 2, we focus on the reverse process: taking synthetic images produced by these models and converting them back into structured network representations.

This step is crucial because real-world applications do not operate on images—they require valid, interpretable packet traces that can be analyzed, replayed, or integrated into downstream tools.

---

### **Your Task for Part 2**

In this part of the assignment, you will:

1. **Convert generated images back into the original nPrint representation.**
   You will follow a scripted pipeline that translates pixel intensities and color channels back into binary header fields, reconstructing the packet-level structure of the trace.

2. **Apply essential post-processing techniques to correct errors introduced by diffusion models.**
   Generated images are rarely perfect—vision models may introduce color drift, pixel misalignment, noise, or structural artifacts.
   You will observe how heuristic correction, formatting enforcement, and reconstruction steps ensure that the converted nPrints become:

   * syntactically valid,
   * structurally consistent,
   * and replayable.

Across this section, your goal is to understand **why the reverse transformation is fragile**, which types of artifacts break reversibility, and how post-processing logic helps repair or compensate for generative errors.

For simplicity of this assignment, we have trainined and generated the images for you. If you have sufficient GPU access and want to try fine-tuning the model and generating the images yourself, feel free to take a look at the public repo (https://github.com/noise-lab/NetDiffusion).


Q4: We have taken the images converted by NetDiffusion and trained a LoRA-fine-tuned Stable Diffusion model (with ControlNet) to generate synthetic traffic images for you. These generated samples are stored in generated_traffic_images/.
Compare these generated images visually with the real images you saw earlier in real_traffic_images/.

Do you notice anything different between the real and generated traffic images? What immediately stands out as potentially problematic if we attempt to convert these generated images back into nPrint format? (Descriptive Only)

When comparing the images a few differences and problems stood out. The generated images capture the same rough layout at a distance but they use a range of colors instead of the real images that just use red, green, and blue. This results in smooth natural-looking gradients and blurry/noisy regions rather than the distinct 3-color encoding nPrint requries. This is problamatic because the image pixels will need to be mapped to a 0, 1, or -1 and a color like yellow will be challenging to categorize. 

Q5: If you were to design a method to convert generated images back into nPrints, how would you do it? Explain your approach and describe how your method addresses the concerns you raised in Q4. (Descriptive only)

Before decoding the pixels, I would run a color corection pass over the generated image. For every pixel, the color will snap to the channel that dominates it using the a nearest color rule. Then, that color will determine the value it represents (red = 1, green = 0, blue = -1). 

This directly fixes the gradiant problem because we force pixels back onto the 3-color palette the nPrint format expects. 

After normalization, each row becomes one packet and each column becomes one nPrint feature. Using this, we would reconstruct the csv format and recover the exact column names and ordering and save as nprint 

Below are a set of pre-written scripts that perform the necessary post-generation augmentation and processing on the synthetic images you obtained from the diffusion model. These scripts handle tasks such as color normalization/augmentation and conversion from generated images back into nPrint format.
(The PCAP step is optional — you may run it if you are interested in observing or replaying the reconstructed traffic.)

In [3]:
# Step 1: Color Augmentation
!python ./scripts/color_processor.py \
  --input_dir="./nd_data/generated_traffic_images" \
  --output_dir="./nd_data/color_corrected_generated_traffic_images"

Processed 100 images.


In [4]:
# Step 2: Image-to-nPrint Conversion
!python ./scripts/image_to_nprint.py \
  --org_nprint ./scripts/column_example.nprint \
  --input_dir ./nd_data/color_corrected_generated_traffic_images \
  --output_dir ./nd_data/generated_nprint

Processing ./nd_data/color_corrected_generated_traffic_images/teams_5.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/teams_5.nprint
Processing ./nd_data/color_corrected_generated_traffic_images/twitch_3.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/twitch_3.nprint
Processing ./nd_data/color_corrected_generated_traffic_images/zoom_7.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/zoom_7.nprint
Processing ./nd_data/color_corrected_generated_traffic_images/zoom_6.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/zoom_6.nprint
Processing ./nd_data/color_corrected_generated_traffic_images/twitch_2.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/twitch_2.nprint
Processing ./nd_data/color_corrected_generated_traffic_images/teams_4.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/teams_4.nprint
Processing ./nd_data/color_corrected_generated_traffic_images/teams_6.png with size 1088 x 1024
Saved ./nd_data/generated_nprint/teams_6

Q6: You may now read through the provided scripts in color_processor.py (color augmentation / normalization) and image_to_nprint.py (image → nPrint reconstruction).
How do the post-processing methods implemented in these scripts compare to the approach you proposed in Q5?
Describe the pros and cons of both methods and highlight any differences in design philosophy, robustness, or assumptions.

The overall process is the same in both approaches: first standardize pixel colors back to pure RGB (red/green/blue), then decode each pixel into the proper nPrint value ({1, 0, -1}) and reconstruct the CSV.

The main implementation difference is that NetDiffusion **splits this into two programs** (`color_processor.py` and `image_to_nprint.py`), while my Q5 design described it as **one unified pipeline**. NetDiffusion also uses **specific fixed thresholds** first (e.g., for red: R > 0, G < 160, B < 100), and only falls back to picking the highest R/G/B channel when those thresholds don't match. My Q5 approach relied mainly on the dominant-channel backup without that first threshold pass.

### NetDiffusion scripts — Pros
- **Modular design:** Color correction and nPrint decoding are separate, so you can inspect corrected images before converting.
- **More robust color fixing:** The two-tier rule (thresholds first, then max-channel fallback) handles drifted pixels better than a single dominant-channel rule.
- **Simple, exact decoding:** `image_to_nprint.py` uses strict RGB → {1, 0, -1} mapping, which is easy to reverse and matches the forward conversion.
- **Uses a reference schema:** Column names come from `column_example.nprint`, so output nPrints stay aligned with the standard format.

### NetDiffusion scripts — Cons
- **Depends on Step 1 working perfectly:** `image_to_nprint.py` only accepts exact colors; pixels that aren't pure R/G/B after correction become `None`/missing values.
- **No validation step:** There is no post-processing to enforce protocol consistency or fix invalid cells after decoding.
- **Slow pixel-by-pixel loops** in both scripts.
- **Hand-tuned thresholds** (160, 100) may not generalize to all generated images.

### My Q5 approach — Pros
- **Single end-to-end pipeline** — easier to reason about as one flow.
- **Includes a validation/cleanup step** I proposed after decoding (clamp invalid values, check basic consistency).
- **More flexible** — not tied to one fixed threshold configuration.

### My Q5 approach — Cons
- **Less specific** about how to handle ambiguous pixels (dominant-channel only, no tuned thresholds).
- **Not implemented/tested** — the provided scripts are a working baseline; mine was a design only.
- **Combining everything in one step** makes it harder to debug whether errors come from color correction or decoding.

### Bottom line
Both methods share the same core idea. NetDiffusion prioritizes **practical modularity and tuned heuristics**; my Q5 design added **validation** but was less specific on color correction. The scripts are stronger on fixing Q4 color drift; my design was stronger on catching errors after conversion — which matters for Part 3 classification.


---

# **Part 3 — Using Real and Synthetic nPrints for Application Classification**

In the previous parts, you learned how network traces can be converted between nPrint and image representations, generated using diffusion models, and reconstructed back into nPrint format.
Now, you will evaluate how useful these generated nPrints are for downstream **machine learning tasks**.

Each nPrint file—whether real or generated—is labeled with the **application** that produced the traffic (e.g., `amazon_1.nprint` means this sample came from Amazon traffic).
In this section, you will treat each **entire nPrint file as a single sample** and build a simple ML pipeline to classify application labels.

To simplify the task, you will restrict your model to use **only the first 3 packets** (3 rows) from each nPrint.
This mimics “early packet classification,” where only the beginning of a flow is available.

---


Q7:

You now have access to both `real_nprints/` and `generated_nprint/`.
Notice that in both directories, files are labeled using the application associated with that nPrint (e.g., `amazon_1.nprint`).
Treat each **nPrint file** as one sample.

**Design an ML pipeline that trains a model using *synthetic nPrints* (from `generated_nprint/`) and evaluates its performance on *real nPrints* (from `real_nprints/`) to predict the correct application label.**

Your pipeline should:

1. Use only the **first 3 packets (first 3 rows)** of each nPrint file as input features.
2. Train a classifier on real data.
3. Test the classifier on generated data.
4. Report how well the classifier performs.

We have already written the script to load the real and generated nprints into DataFrame for you.

In [5]:
import os
import glob
import numpy as np
import pandas as pd

# ---------------------------------------------------------------
# Safe conversion for nPrint cell
# ---------------------------------------------------------------
def safe_convert(x):
    if pd.isna(x):
        return 0
    x = str(x).strip()

    if x in ["0", "1", "-1"]:
        return int(x)

    if all(c in "01" for c in x) and len(x) <= 16:
        return int(x, 2)

    if x.lstrip("-").isdigit():
        return int(x)

    return 0


# ---------------------------------------------------------------
# Get original column names from a reference nPrint
# ---------------------------------------------------------------
def get_original_columns(example_path="nd_data/real_nprints"):
    first_file = glob.glob(os.path.join(example_path, "*.nprint"))[0]

    df = pd.read_csv(first_file)

    # Drop index column if present (like "Unnamed: 0")
    if df.columns[0].lower().startswith("unnamed"):
        df = df.drop(df.columns[0], axis=1)

    return list(df.columns)


# ---------------------------------------------------------------
# Load nPrint → first 3 rows → flatten with prefixed column names
# ---------------------------------------------------------------
def load_nprint_with_colnames(path, base_cols, num_rows=3):
    df = pd.read_csv(path, dtype=str, low_memory=False)

    # Drop "Unnamed: 0" if present
    if df.columns[0].lower().startswith("unnamed"):
        df = df.drop(df.columns[0], axis=1)

    df = df.iloc[:num_rows, :]            # first 3 packets  
    df = df.map(safe_convert)             # clean convert  

    # Build prefixed column names
    pkt_cols = []
    for pkt in range(1, num_rows + 1):
        pkt_cols.extend([f"pkt{pkt}_{c}" for c in base_cols])

    # Flatten 3×columns into 1 vector
    flat = df.values.flatten()

    return flat, pkt_cols


# ---------------------------------------------------------------
# Load entire directory into a DataFrame (with labels)
# ---------------------------------------------------------------
def load_directory_as_df(directory, base_cols):
    rows = []
    labels = []
    colnames_set = None

    for path in glob.glob(os.path.join(directory, "*.nprint")):
        label = os.path.basename(path).split("_")[0]

        flat, cn = load_nprint_with_colnames(path, base_cols)
        rows.append(flat)
        labels.append(label)

        if colnames_set is None:
            colnames_set = cn   # only set once

    df = pd.DataFrame(rows, columns=colnames_set)
    df["label"] = labels
    return df


# ---------------------------------------------------------------
# FINAL: Load real + synthetic DataFrames
# ---------------------------------------------------------------
base_cols = get_original_columns("nd_data/real_nprints")

df_synth = load_directory_as_df("nd_data/generated_nprint", base_cols)
df_real  = load_directory_as_df("nd_data/real_nprints", base_cols)

print("Synthetic DF:", df_synth.shape)
print(df_synth.head())

print("\nReal DF:", df_real.shape)
print(df_real.head())


Synthetic DF: (100, 3265)
   pkt1_ipv4_ver_0  pkt1_ipv4_ver_1  pkt1_ipv4_ver_2  pkt1_ipv4_ver_3  \
0                1                0                0                0   
1                1                0                0                0   
2                0                0                0                0   
3                0                0                0                0   
4                1                1                1                1   

   pkt1_ipv4_hl_0  pkt1_ipv4_hl_1  pkt1_ipv4_hl_2  pkt1_ipv4_hl_3  \
0               0               0               0               1   
1               0               0               0               1   
2               0               0               0               0   
3               0               0               0               0   
4               1               1               1               1   

   pkt1_ipv4_tos_0  pkt1_ipv4_tos_1  ...  pkt3_icmp_roh_23  pkt3_icmp_roh_24  \
0                0                0  ...

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

# NOTE: Q7 has conflicting instructions:
#   - Bold text: train on SYNTHETIC, test on REAL
#   - Numbered bullets: train on REAL, test on SYNTHETIC
# We run BOTH experiments below and compare results.

feature_cols = [c for c in df_real.columns if c != "label" and c in df_synth.columns]

X_real = df_real[feature_cols].fillna(0)
y_real = df_real["label"]
X_synth = df_synth[feature_cols].fillna(0)
y_synth = df_synth["label"]

print(f"Real samples:      {len(X_real)}")
print(f"Synthetic samples: {len(X_synth)}")
print(f"Features/sample:   {len(feature_cols)}  (3 packets × header bits)")
print(f"Classes:           {sorted(y_real.unique())}")

Real samples:      200
Synthetic samples: 100
Features/sample:   3264  (3 packets × header bits)
Classes:           ['amazon', 'facebook', 'instagram', 'meet', 'netflix', 'teams', 'twitch', 'twitter', 'youtube', 'zoom']


In [11]:
# Experiment A (bold Q7 instruction): train synthetic → test real
clf_synth_to_real = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf_synth_to_real.fit(X_synth, y_synth)

y_pred_a = clf_synth_to_real.predict(X_real)
acc_a = accuracy_score(y_real, y_pred_a)

print("=" * 60)
print("Experiment A: train=synthetic, test=real")
print(f"Accuracy: {acc_a:.3f}\n")
print(classification_report(y_real, y_pred_a, zero_division=0))

Experiment A: train=synthetic, test=real
Accuracy: 0.280

              precision    recall  f1-score   support

      amazon       0.00      0.00      0.00        20
    facebook       0.00      0.00      0.00        20
   instagram       0.67      0.30      0.41        20
        meet       0.39      0.35      0.37        20
     netflix       0.80      0.20      0.32        20
       teams       0.27      0.95      0.42        20
      twitch       0.00      0.00      0.00        20
     twitter       0.22      1.00      0.36        20
     youtube       0.00      0.00      0.00        20
        zoom       0.00      0.00      0.00        20

    accuracy                           0.28       200
   macro avg       0.23      0.28      0.19       200
weighted avg       0.23      0.28      0.19       200



In [12]:
# Experiment B (numbered bullets): train real → test synthetic
clf_real_to_synth = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf_real_to_synth.fit(X_real, y_real)

y_pred_b = clf_real_to_synth.predict(X_synth)
acc_b = accuracy_score(y_synth, y_pred_b)

print("=" * 60)
print("Experiment B: train=real, test=synthetic")
print(f"Accuracy: {acc_b:.3f}\n")
print(classification_report(y_synth, y_pred_b, zero_division=0))

Experiment B: train=real, test=synthetic
Accuracy: 0.350

              precision    recall  f1-score   support

      amazon       0.00      0.00      0.00        10
    facebook       0.00      0.00      0.00        10
   instagram       0.00      0.00      0.00        10
        meet       0.27      1.00      0.43        10
     netflix       0.36      0.90      0.51        10
       teams       0.00      0.00      0.00        10
      twitch       0.00      0.00      0.00        10
     twitter       0.00      0.00      0.00        10
     youtube       0.56      1.00      0.71        10
        zoom       0.40      0.60      0.48        10

    accuracy                           0.35       100
   macro avg       0.16      0.35      0.21       100
weighted avg       0.16      0.35      0.21       100



In [13]:
# Baseline: how well does the classifier work when both train and test are real?
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(
    X_real, y_real, test_size=0.2, random_state=42, stratify=y_real
)
baseline = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
baseline.fit(X_tr, y_tr)
baseline_acc = baseline.score(X_val, y_val)

print("=" * 60)
print(f"Baseline (real train/val split): {baseline_acc:.3f}")
print("\nSummary:")
print(f"  A) synthetic → real:  {acc_a:.3f}")
print(f"  B) real → synthetic:  {acc_b:.3f}")
print(f"  Baseline (real only): {baseline_acc:.3f}")

Baseline (real train/val split): 0.950

Summary:
  A) synthetic → real:  0.280
  B) real → synthetic:  0.350
  Baseline (real only): 0.950


Both cross-domain experiments (28% and 35%) perform far below the real-only baseline (95%). This suggests the generate→image→reconstruct pipeline loses too much application-specific structure for synthetic nPrints to be useful for training or evaluation. Improving accuracy would require better synthetic generation and reconstruction (Parts 1–2), not just a different classifier in Q7.

### Q7 Results

Q7 had conflicting instructions (bold text vs numbered bullets), so I ran both:

1. **Train synthetic → test real** (bold instruction): answers whether synthetic nPrints can replace real training data.
2. **Train real → test synthetic** (numbered bullets): answers how faithful reconstructed synthetic nPrints are.

Both experiments perform much worse than the real-only baseline (~95%), which shows the generate → image → reconstruct pipeline still loses application-specific header structure. Synthetic nPrints are not yet good enough to train a reliable classifier for real traffic, and reconstructed traces don't match real traces well enough for a real-trained model to classify them accurately.

Cursor used to assist method creation and assignment clarificaiton. Also to add clarity and detail to the written responses 
Model used: Composer 2.5 